Generate an UMAP plot for VITEK, Phoenix and unclassified.

In [ ]:
import pandas as pd
import umap
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder

# Load samples
df_vitek = pd.read_csv("EKP_vitek_parsed_cleaned.csv")
df_phoenix = pd.read_csv("EKP_phoenix_parsed_cleaned.csv")
df_na = pd.read_csv("EKP_nan_parsed_cleaned.csv")

#Clean samples from non EKP
df_vitek = df_vitek[df_vitek["Organism_Code"].str.contains("EKP", case=False, na=False)]
df_phoenix = df_phoenix[
    df_phoenix["Organism_Code"].str.contains("EKP", case=False, na=False)
]
df_na = df_na[df_na["Organism_Code"].str.contains("EKP", case=False, na=False)]
df_na = df_na.dropna(subset=df_na.columns[2:], how="all")


# Device label
df_vitek["device"] = "VITEK"
df_phoenix["device"] = "PHOENIX"
df_na["device"] = "Unclassified"


# Combine datasets
df = pd.concat(
    [
        df_vitek,
        df_phoenix,
        df_na,
    ],
    ignore_index=True,
)

# Encoding
print("Combined shape:", df.shape)
y = df["device"]
exclude_cols = ["Sample_ID_IfH", "device", "Organism_Code"]
X = df.drop(columns=exclude_cols)
X = X.fillna("MISSING")
X = X.astype(str)

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_encoded = pd.DataFrame(encoder.fit_transform(X), columns=X.columns, index=X.index)

# Create UMAP
reducer = umap.UMAP(
    n_neighbors=10,
    min_dist=0.5,
    metric="hamming",
    random_state=42,
)
embedding = reducer.fit_transform(X_encoded)

# Plot
plt.figure(figsize=(10, 7))

for device in y.unique():
    idx = y == device
    plt.scatter(embedding[idx, 0], embedding[idx, 1], label=device, alpha=0.6, s=10)

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend()
plt.savefig("umap_Unclassified.pdf", format="pdf", bbox_inches="tight")
plt.show()

/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Combined shape: (4285, 83)


/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
